# 在线策略蒸馏（OPD）

> 我们已经知道，传统蒸馏的做法是让大模型（Teacher）写一批标准答案，小模型（Student）背诵。但小模型在训练时从没见过自己犯错的状态——它只见过 Teacher 的完美输出。一旦进入推理阶段开始自己生成，遇到自己不熟悉的分布，质量就会下降。
>
> 这一节，我们深入理解 On-Policy Distillation（OPD）的核心原理和实现：不再用 Teacher 的静态输出做训练数据，而是让 Student 自己生成、Teacher 实时打分、Student 用这个反馈来自我改进。

OPD 和传统蒸馏的本质区别在训练数据的来源。传统蒸馏的 data 是 Teacher 自己写的（off-policy），无论 Student 怎么更新，训练数据都不变。OPD 的 data 是 Student 生成的（on-policy）——Student 每轮先对 prompt 输出一个回答，Teacher 在相同的 prompt 上给出自己的参考回答，Teacher 还逐句比较两个回答给出改进意见，Student 用这个信号更新自己，下一轮重新生成。这个过程和 RLHF 的 PPO 很像——Student 是 Actor，Teacher 同时扮演 Reward Model 和 Reference Model。但 OPD 不需要显式训练 Reward Model（直接用 Teacher 做评判），也不需要 constraint optimization（直接做 SFT 式的监督训练）。下面从 off-policy vs on-policy 的根本区别出发，逐步展开 OPD 的训练循环。

In [1]:
import numpy as np

np.random.seed(42)

## 1. 知识蒸馏回顾

你有一个大模型（Teacher，老师）和一个小模型（Student，学生）。
目标是让小学生也能答出大学生水平的题。

```
Teacher (GPT-4, 1.8T 参数):  输入 "1+1=?" → 输出 "2" ✅
Student (小模型, 0.5B 参数): 输入 "1+1=?" → 输出 "3" ❌

蒸馏目标: 让 Student 也输出 "2"
```

怎么做？两种思路：
- **离线蒸馏**：Teacher 先写一本「标准答案集」，Student 照着背
- **在线蒸馏（OPD）**：Student 先自己做题，Teacher 现场批改

## 2. 四种训练方式对比

只看两个问题就能分清所有方法：

1. **训练时的前缀（prefix）是谁写的？**
2. **学习目标（target）是谁给的？**

| 方法 | prefix 来源 | target 来源 | 类比 |
|:---|:---|:---|:---|
| **SFT** | 数据集标准答案 | 数据集标准答案 | 背课本 |
| **离线蒸馏** | Teacher 写好的固定前缀 | Teacher 的概率分布 | 背老师写的范文 |
| **OPD** | **Student 自己生成的前缀** | Teacher 在这个学生前缀上的分布 | 学生自己写，老师批改 |
| **OPSD** | **Student 自己生成的前缀** | 同一个模型的开卷版本 | 学生闭卷做题，开卷的自己来教 |

核心区别就是 prefix 那一列。OPD 和 OPSD 的 prefix 是学生自己写的，其他都是别人写好的。

## 3. 问题根源：Exposure Bias（暴露偏差）

这是理解 OPD 为什么必要的**最关键概念**。

```
训练时（SFT / 离线蒸馏）:
  前缀永远是标准答案的前半段
  比如: "我昨天去了" → 学预测 "学校"
  学生看到的前缀都是语法正确、语义通顺的

推理时（学生自己生成）:
  学生可能写出离谱的前缀
  比如: "我昨天很公园" → 现在要预测下一个词？
  学生从没见过 "我昨天很公园" 这种前缀！
  → 完全不知道怎么接 → 崩溃
```

**这就是 Exposure Bias**：训练时站在「正确轨道」上，推理时一旦走偏就进入未知领域。

就像你只在平坦的高速公路上练过车，第一次开到乡间土路就慌了。

In [2]:
# 模拟词汇表
vocab = {"我": 0, "昨天": 1, "去了": 2, "学校": 3, "公园": 4, "超市": 5, "很": 6, "开心": 7, "无聊": 8, "累": 9}
id2token = {v: k for k, v in vocab.items()}
vocab_size = len(vocab)

def softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    exp_logits = np.exp(logits - logits.max())
    return exp_logits / exp_logits.sum()

def log_softmax(logits):
    logits = np.array(logits, dtype=np.float64)
    m = logits.max()
    return logits - m - np.log(np.exp(logits - m).sum())

# 模拟学生模型：根据输入序列输出 logits
def student_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 7 + 42)
    return rng.randn(vocab_size) * 2

# 模拟老师模型：更聪明、更确定
def teacher_model(input_ids):
    rng = np.random.RandomState(sum(input_ids) * 13 + 7)
    return rng.randn(vocab_size) * 1.2

print("词汇表和模型模拟函数准备完毕！")

词汇表和模型模拟函数准备完毕！


In [3]:
# 直观演示 Exposure Bias：用真实模型模拟训练 vs 推理的差异
print("=== Exposure Bias 演示 ===")
print()

# 训练时的前缀：标准答案的前半段
train_prefix = [0, 1, 2]  # "我昨天去了"
print(f"训练时的前缀: {[id2token[i] for i in train_prefix]} ← 数据集给的，语法正确")
t_logits = teacher_model(train_prefix)
t_probs = softmax(t_logits)
top3 = np.argsort(t_probs)[-3:][::-1]
print("  学生在这个前缀下被训练，老师最看好的 3 个词:")
for idx in top3:
    print(f"    '{id2token[idx]}': {t_probs[idx]:.3f}")
print()

# 推理时：学生自己生成，可能走偏
print("推理时，学生自回归生成:")
np.random.seed(99)
student_sequence = [0]  # "我"
for step in range(3):
    s_logits = student_model(student_sequence)
    s_probs = softmax(s_logits)
    next_token = np.random.choice(vocab_size, p=s_probs)
    student_sequence.append(next_token)
    print(f"  Step {step+1}: 输入 {[id2token[i] for i in student_sequence[:-1]]} "
          f"→ 采样到 '{id2token[next_token]}' (概率={s_probs[next_token]:.3f})")
print()

# 检查学生生成的离谱前缀下模型的困惑程度
weird_prefix = student_sequence[:-1]
s_logits_weird = student_model(weird_prefix)
s_probs_weird = softmax(s_logits_weird)
entropy = -np.sum(s_probs_weird * np.log(s_probs_weird + 1e-10))
print(f"学生生成的前缀: {[id2token[i] for i in weird_prefix]}")
print(f"  模型输出熵: {entropy:.3f} (越高越不确定)")
print(f"  最高概率的词: '{id2token[np.argmax(s_probs_weird)]}' ({np.max(s_probs_weird):.3f})")
print()
print("→ 学生从未在训练中见过这种前缀！输出非常不确定。")
print("→ Exposure Bias：训练和推理的 prefix 分布不一致。")

=== Exposure Bias 演示 ===

训练时的前缀: ['我', '昨天', '去了'] ← 数据集给的，语法正确
  学生在这个前缀下被训练，老师最看好的 3 个词:
    '昨天': 0.257
    '累': 0.164
    '去了': 0.157

推理时，学生自回归生成:
  Step 1: 输入 ['我'] → 采样到 '很' (概率=0.386)
  Step 2: 输入 ['我', '很'] → 采样到 '去了' (概率=0.518)
  Step 3: 输入 ['我', '很', '去了'] → 采样到 '去了' (概率=0.190)

学生生成的前缀: ['我', '很', '去了']
  模型输出熵: 1.019 (越高越不确定)
  最高概率的词: '昨天' (0.682)

→ 学生从未在训练中见过这种前缀！输出非常不确定。
→ Exposure Bias：训练和推理的 prefix 分布不一致。


## 4. OPD 的解决方案：在自己的轨迹上学习

OPD 的核心就三步：

```
Step 1: Student 自己生成一段回答（rollout）
  Prompt: "我昨天" → Student 生成: "我昨天去了公园很开心"

Step 2: Teacher 在 Student 的每个 prefix 上给反馈
  位置 2: prefix=[我, 昨天], Student 写了"去了" → Teacher: "还行"
  位置 3: prefix=[我, 昨天, 去了], Student 写了"公园" → Teacher: "可以"
  位置 4: prefix=[我, 昨天, 去了, 公园], Student 写了"很" → Teacher: "不太对"

Step 3: 根据 Teacher 的反馈更新 Student
  Teacher 认可的词 → 提高概率
  Teacher 不认可的词 → 降低概率
```

关键：Student 是在**自己真实写出来的轨迹**上被纠正的。
即使写出了「我昨天很公园」这种离谱前缀，Teacher 也会在这个离谱前缀上告诉它下一步该怎么走。
这样推理时遇到同样的离谱前缀，Student 就知道怎么办了。

In [4]:
# 演示 OPD 的核心：Student 自己 rollout，Teacher 在 Student 的 prefix 上打分
print("=== OPD 核心流程演示 ===")
print()

prompt = [0, 1]  # "我昨天"
sequence = prompt.copy()
temperature = 1.0

print(f"Prompt: {[id2token[i] for i in prompt]}")
print()

# Step 1: Student rollout（学生自己写）
print("Step 1: Student 自回归生成")
for step in range(3):
    logits = student_model(sequence)
    probs = softmax(logits / temperature)
    next_token = np.random.choice(vocab_size, p=probs)
    sequence.append(next_token)
    print(f"  前缀={[id2token[i] for i in sequence[:-1]]} → 采样 {id2token[next_token]}")

print(f"\n生成序列: {[id2token[i] for i in sequence]}")
print()

# Step 2: Teacher 在每个位置打分
print("Step 2: Teacher 在 Student 自己的 prefix 上评估")
for t in range(len(prompt), len(sequence)):
    prefix = sequence[:t]
    sampled_token = sequence[t]
    
    s_logits = student_model(prefix)
    t_logits = teacher_model(prefix)
    s_logp = log_softmax(s_logits)[sampled_token]
    t_logp = log_softmax(t_logits)[sampled_token]
    
    advantage = t_logp - s_logp
    
    print(f"  位置 {t}: prefix={[id2token[i] for i in prefix]}")
    print(f"    Student 写了 '{id2token[sampled_token]}', 学生logp={s_logp:.3f}, 老师logp={t_logp:.3f}")
    print(f"    advantage = {advantage:+.3f} → {'✅ 加分' if advantage > 0 else '❌ 扣分'}")

print()
print("关键：不管 Student 写的是什么离谱内容，Teacher 都在这个真实的 prefix 上给反馈！")

=== OPD 核心流程演示 ===

Prompt: ['我', '昨天']

Step 1: Student 自回归生成
  前缀=['我', '昨天'] → 采样 去了
  前缀=['我', '昨天', '去了'] → 采样 很
  前缀=['我', '昨天', '去了', '很'] → 采样 公园

生成序列: ['我', '昨天', '去了', '很', '公园']

Step 2: Teacher 在 Student 自己的 prefix 上评估
  位置 2: prefix=['我', '昨天']
    Student 写了 '去了', 学生logp=-3.118, 老师logp=-2.268
    advantage = +0.850 → ✅ 加分
  位置 3: prefix=['我', '昨天', '去了']
    Student 写了 '很', 学生logp=-2.144, 老师logp=-2.923
    advantage = -0.779 → ❌ 扣分
  位置 4: prefix=['我', '昨天', '去了', '很']
    Student 写了 '公园', 学生logp=-0.636, 老师logp=-5.154
    advantage = -4.518 → ❌ 扣分

关键：不管 Student 写的是什么离谱内容，Teacher 都在这个真实的 prefix 上给反馈！


## 5. 数学本质：Forward KL vs Reverse KL

离线蒸馏和 OPD 的差异，可以用 KL 散度的**方向**来理解。

KL 散度衡量两个分布的「距离」，但它**不对称**：

$$D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$$

就像「从家到学校」和「从学校到家」虽然路程一样，但上下坡的感受完全不同。

**Forward KL（离线蒸馏）**：

$$D_{KL}(P_{Teacher} \| P_{Student})$$

- 期望在 Teacher 的分布下取：Teacher 觉得重要的地方，Student 必须学好
- 行为：**Mode-Covering** — Teacher 的所有「模式」Student 都要覆盖
- 问题：小模型容量不够，强行覆盖所有模式 → 每个都学个皮毛 → 输出「平均化」

**Reverse KL（OPD）**：

$$D_{KL}(P_{Student} \| P_{Teacher})$$

- 期望在 Student 的分布下取：Student 自己常去的地方，才需要向 Teacher 对齐
- 行为：**Mode-Seeking** — Student 只需要找到 Teacher 认可的、自己能稳定生成的模式
- 优势：小模型可以「专攻一门」，不用面面俱到

In [5]:
# 用真实概率分布演示 Forward KL vs Reverse KL
print("=== Forward KL vs Reverse KL 直觉 ===")
print()

# 模拟 Teacher 的三种风格概率分布
teacher_probs = np.array([0.40, 0.35, 0.25])
styles = ["学术严谨风", "幽默风趣风", "简洁直接风"]

print("Teacher 认为好答案有三种风格:")
for i, (style, prob) in enumerate(zip(styles, teacher_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print()

# Forward KL (离线蒸馏): Student 被要求覆盖所有模式
# 用一个"平均化"的 Student 分布来展示
student_avg_probs = np.array([0.33, 0.33, 0.34])  # 均匀化
forward_kl = np.sum(teacher_probs * (np.log(teacher_probs + 1e-10) - np.log(student_avg_probs + 1e-10)))

print("Forward KL (离线蒸馏) — Student 试图覆盖所有模式:")
for i, (style, prob) in enumerate(zip(styles, student_avg_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Forward KL = {forward_kl:.4f}")
print(f"  → 输出变成四不像：'根据相关研究，哈哈，简单来说...'")
print()

# Reverse KL (OPD): Student 选择自己擅长的模式深耕
student_spec_probs = np.array([0.05, 0.85, 0.10])  # 专攻幽默风
reverse_kl = np.sum(student_spec_probs * (np.log(student_spec_probs + 1e-10) - np.log(teacher_probs + 1e-10)))

print("Reverse KL (OPD) — Student 专攻自己擅长的模式:")
for i, (style, prob) in enumerate(zip(styles, student_spec_probs)):
    bar = "█" * int(prob * 40)
    print(f"  {style}: {prob:.0%} {bar}")
print(f"  Reverse KL = {reverse_kl:.4f}")
print(f"  → 稳定、高质量的幽默回答")
print()

print(f"对比: Forward KL ({forward_kl:.4f}) vs Reverse KL ({reverse_kl:.4f})")
print("OPD 对小模型更友好：不需要面面俱到，专攻一门就够了！")

=== Forward KL vs Reverse KL 直觉 ===

Teacher 认为好答案有三种风格:
  学术严谨风: 40% ████████████████
  幽默风趣风: 35% ██████████████
  简洁直接风: 25% ██████████

Forward KL (离线蒸馏) — Student 试图覆盖所有模式:
  学术严谨风: 33% █████████████
  幽默风趣风: 33% █████████████
  简洁直接风: 34% █████████████
  Forward KL = 0.0207
  → 输出变成四不像：'根据相关研究，哈哈，简单来说...'

Reverse KL (OPD) — Student 专攻自己擅长的模式:
  学术严谨风: 5% ██
  幽默风趣风: 85% ██████████████████████████████████
  简洁直接风: 10% ████
  Reverse KL = 0.5586
  → 稳定、高质量的幽默回答

对比: Forward KL (0.0207) vs Reverse KL (0.5586)
OPD 对小模型更友好：不需要面面俱到，专攻一门就够了！


## 6. OPSD：不需要外部 Teacher 的 OPD

OPD 需要一个大 Teacher 模型，但 OPSD（On-Policy Self-Distillation）不需要——**Teacher 和 Student 是同一个模型，只是 Teacher 多看到了一些信息。**

```
OPSD 的核心设定:

Student (闭卷):  只看到题目
  → 自己写答案

Teacher (开卷):  同一个模型，但看到了标准答案/解题步骤
  → 在 Student 写的 prefix 上给出「如果我知道答案，我下一步会怎么写」

训练目标: 让闭卷的 Student 靠近开卷的 Teacher
→ 把「看答案才知道」的能力内化到「不看答案也能写」
```

**直觉**：做数学题时，看着答案你觉得「哦原来这样」，但合上答案就忘了。
OPSD 就是反复练「看着答案的自己」教「合上答案的自己」。

**局限**：如果 Teacher 多看的信息在测试时永远拿不到（比如每道题独有的标准解法），
OPSD 可能学到的只是「平均化的做题策略」，而不是真正的推理能力。
所以 OPSD 更适合把**共享规则**（格式偏好、系统提示）内化，而不是把**每道题独有的信息**内化。

## 7. 三种信号粒度：老师告诉你多少信息

Teacher 给反馈时，可以给不同「粒度」的信息：

| 粒度 | 老师告诉你什么 | 信息量 | 计算量 |
|:---|:---|:---|:---|
| **sampled-token** | 只告诉你「你选的这个词好不好」 | 最少 | 最小 |
| **top-k** | 告诉你「我心目中最好的 K 个词是哪些」 | 中等 | 中等 |
| **full-vocab** | 告诉你「整个词表每个词的概率」 | 最多 | 最大（32000 维！）|

```
sampled-token:  学生写了"公园" → Teacher: "还行，给 6 分"
top-k (k=5):    学生写了"公园" → Teacher: "最好的 5 个是: 学校(9分) 公园(6分) 超市(4分) 家(3分) 商场(2分)"
full-vocab:      学生写了"公园" → Teacher: "32000 个词各多少分..."
```

In [6]:
# 对比三种粒度的 Teacher 反馈
prefix_demo = [0, 1, 2]  # "我昨天去了"

print("=== 三种信号粒度对比 ===")
print()

# Sampled-token: 只看选中的词
t_logits = teacher_model(prefix_demo)
t_logprobs = log_softmax(t_logits)
sampled_token = 3  # "学校"

print(f"1. sampled-token:")
print(f"   老师只告诉你 '{id2token[sampled_token]}' 的分数: {t_logprobs[sampled_token]:.3f}")
print(f"   其他 9 个词的情况你不知道")
print()

# Top-k: 看老师最看好的 K 个
k = 5
topk_idx = np.argsort(t_logprobs)[-k:][::-1]
print(f"2. top-{k}:")
for idx in topk_idx:
    print(f"   老师认为 '{id2token[idx]}' 分数: {t_logprobs[idx]:.3f}")
print()

# Full-vocab
print(f"3. full-vocab:")
print(f"   老师告诉你全部 {vocab_size} 个词的概率分布")
all_probs = softmax(t_logits)
for i, tok in id2token.items():
    bar = "█" * int(all_probs[i] * 40)
    print(f"   {tok:4s}: {all_probs[i]:.4f}  {bar}")

=== 三种信号粒度对比 ===

1. sampled-token:
   老师只告诉你 '学校' 的分数: -3.794
   其他 9 个词的情况你不知道

2. top-5:
   老师认为 '昨天' 分数: -1.357
   老师认为 '累' 分数: -1.806
   老师认为 '去了' 分数: -1.849
   老师认为 '我' 分数: -2.133
   老师认为 '公园' 分数: -2.340

3. full-vocab:
   老师告诉你全部 10 个词的概率分布
   我   : 0.1185  ████
   昨天  : 0.2573  ██████████
   去了  : 0.1575  ██████
   学校  : 0.0225  
   公园  : 0.0963  ███
   超市  : 0.0475  █
   很   : 0.0538  ██
   开心  : 0.0298  █
   无聊  : 0.0525  ██
   累   : 0.1643  ██████


## 8. 当只能用 sampled-token：KL 估计器

sampled-token 最省计算，但我们只有一个 token 的信息，却要估计整个分布的 KL 散度。怎么办？

真正的 Reverse KL 需要整个分布：

$$KL(P_S \| P_T) = \sum_i P_S(i) \cdot \log\frac{P_S(i)}{P_T(i)}$$

但现在我们只有一个采样出来的 token `y`。三种估计方法：

| 估计器 | 公式 | 特点 |
|:---|:---|:---|
| **k1** | $\log P_S(y) - \log P_T(y)$ | 无偏，但方差大，可为负 |
| **k2** | $\frac{1}{2}(\log P_S(y) - \log P_T(y))^2$ | 永远正数，有偏但稳定 |
| **k3** | $\frac{P_T(y)}{P_S(y)} - \log\frac{P_T(y)}{P_S(y)} - 1$ | 无偏 + 非负 + 低方差，推荐！ |

In [7]:
def k1_estimator(s_logp, t_logp):
    """直接差值：无偏但方差大，可为负"""
    return s_logp - t_logp

def k2_estimator(s_logp, t_logp):
    """平方近似：有偏但永远正数，方差小"""
    diff = s_logp - t_logp
    return 0.5 * diff * diff

def k3_estimator(s_logp, t_logp):
    """k3：无偏 + 非负 + 低方差（推荐）"""
    ratio = t_logp - s_logp  # log(P_T / P_S)
    r = np.exp(ratio)         # P_T / P_S
    return r - ratio - 1

# 对比三种估计器
print("=== 三种 KL 估计器对比 ===")
print(f"{'学生logp':>10} {'老师logp':>10} {'k1':>10} {'k2':>10} {'k3':>10}  说明")
print("-" * 65)

cases = [
    (-0.5, -0.5, "完全一致"),
    (-1.0, -0.5, "老师更看好"),
    (-0.5, -1.0, "学生更看好"),
    (-2.0, -0.5, "差距大-老师看好"),
    (-0.5, -2.0, "差距大-学生看好"),
    (-3.0, -0.1, "极端差距"),
]

for s_lp, t_lp, desc in cases:
    k1 = k1_estimator(s_lp, t_lp)
    k2 = k2_estimator(s_lp, t_lp)
    k3 = k3_estimator(s_lp, t_lp)
    print(f"{s_lp:>+10.2f} {t_lp:>+10.2f} {k1:>+10.4f} {k2:>10.4f} {k3:>10.4f}  ← {desc}")

print()
print("k1 可为负 → 不适合直接当 loss")
print("k2 永远正数 → 有偏但稳定")
print("k3 永远正数 → 无偏且稳定，推荐使用")

=== 三种 KL 估计器对比 ===
    学生logp     老师logp         k1         k2         k3  说明
-----------------------------------------------------------------
     -0.50      -0.50    +0.0000     0.0000     0.0000  ← 完全一致
     -1.00      -0.50    -0.5000     0.1250     0.1487  ← 老师更看好
     -0.50      -1.00    +0.5000     0.1250     0.1065  ← 学生更看好
     -2.00      -0.50    -1.5000     1.1250     1.9817  ← 差距大-老师看好
     -0.50      -2.00    +1.5000     1.1250     0.7231  ← 差距大-学生看好
     -3.00      -0.10    -2.9000     4.2050    14.2741  ← 极端差距

k1 可为负 → 不适合直接当 loss
k2 永远正数 → 有偏但稳定
k3 永远正数 → 无偏且稳定，推荐使用


## 9. 完整 OPD 训练流程（串起来）

前面分别讲了 OPD 的各个组件：Student 自生成、Teacher 打分、Teacher 批改、Student 更新。现在把它们串成一次完整的训练迭代。

一次迭代的步骤是：取一个 prompt → Student 自己生成回答 → Teacher 对同一个 prompt 生成参考回答 → Teacher 比较两个回答给出逐句批注 → 将批注整理成 SFT 训练数据 → 更新 Student。下一轮迭代，Student 从更新后的参数出发重新生成，Teacher 再次批改。这个循环的关键在于：Student 每次都在自己犯错后的新状态下被纠正，而不是背别人的标准答案。

下面把整个循环写成一个可运行的训练 loop。

In [8]:
def simulate_opd_training(prompt_ids, num_gen_tokens=3, mode="sampled_token_k3", topk=5):
    """模拟一次完整的 OPD 训练步骤"""
    
    # Step 1: Student rollout（学生自己生成）
    sequence = list(prompt_ids).copy()
    temp = 1.0
    
    print("Step 1: Rollout（学生自回归生成）")
    print(f"  Prompt: {[id2token[i] for i in prompt_ids]}")
    for step in range(num_gen_tokens):
        logits = student_model(sequence)
        probs = softmax(logits / temp)
        next_token = np.random.choice(vocab_size, p=probs)
        sequence.append(next_token)
    
    generated = sequence[len(prompt_ids):]
    print(f"  生成: {[id2token[i] for i in generated]}")
    
    # Step 2: 每个位置计算 OPD loss
    print(f"\nStep 2: 计算 OPD 损失 (模式: {mode})")
    total_loss = 0
    
    for t in range(len(prompt_ids), len(sequence)):
        prefix = sequence[:t]
        sampled_token = sequence[t]
        
        if mode == "sampled_token_k3":
            s_logp = log_softmax(student_model(prefix))[sampled_token]
            t_logp = log_softmax(teacher_model(prefix))[sampled_token]
            loss = k3_estimator(s_logp, t_logp)
            print(f"  位置{t}: '{id2token[sampled_token]}' | s_logp={s_logp:.3f} t_logp={t_logp:.3f} | k3={loss:.4f}")
        
        elif mode == "topk_rkl":
            t_logp = log_softmax(teacher_model(prefix))
            topk_idx = np.argsort(t_logp)[-topk:][::-1]
            t_renorm = softmax(t_logp[topk_idx])
            s_logp = log_softmax(student_model(prefix))
            s_renorm = softmax(s_logp[topk_idx])
            loss = np.sum(s_renorm * (np.log(s_renorm + 1e-10) - np.log(t_renorm + 1e-10)))
            print(f"  位置{t}: top-{topk}={[id2token[i] for i in topk_idx]} | RKL={loss:.4f}")
        
        total_loss += loss
    
    avg_loss = total_loss / len(generated)
    print(f"\n总损失: {total_loss:.4f} | 平均: {avg_loss:.4f}")
    print(f"→ 反传梯度，更新 Student → 下一轮 rollout → 循环")
    return avg_loss

np.random.seed(42)
print("### sampled_token + k3 模式 ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="sampled_token_k3")

### sampled_token + k3 模式 ###
Step 1: Rollout（学生自回归生成）
  Prompt: ['我', '昨天']
  生成: ['学校', '超市', '超市']

Step 2: 计算 OPD 损失 (模式: sampled_token_k3)
  位置2: '学校' | s_logp=-0.803 t_logp=-5.509 | k3=3.7149
  位置3: '超市' | s_logp=-0.685 t_logp=-4.153 | k3=2.4994
  位置4: '超市' | s_logp=-2.903 t_logp=-4.291 | k3=0.6372

总损失: 6.8515 | 平均: 2.2838
→ 反传梯度，更新 Student → 下一轮 rollout → 循环


np.float64(2.283848208244739)

In [9]:
np.random.seed(42)
print("### top-k Reverse KL 模式 ###")
simulate_opd_training([0, 1], num_gen_tokens=3, mode="topk_rkl", topk=5)

### top-k Reverse KL 模式 ###
Step 1: Rollout（学生自回归生成）
  Prompt: ['我', '昨天']
  生成: ['学校', '超市', '超市']

Step 2: 计算 OPD 损失 (模式: topk_rkl)
  位置2: top-5=['很', '我', '超市', '无聊', '累'] | RKL=1.4217
  位置3: top-5=['开心', '公园', '昨天', '去了', '无聊'] | RKL=0.5603
  位置4: top-5=['学校', '开心', '无聊', '我', '昨天'] | RKL=0.3147

总损失: 2.2966 | 平均: 0.7655
→ 反传梯度，更新 Student → 下一轮 rollout → 循环


np.float64(0.7655468551277211)

## 10. OPD 为什么现在才火

OPD 的思想不是新的，但过去很难落地——因为要在训练过程中反复做 Student rollout + Teacher 打分，工程复杂度远高于普通 SFT。

近两年三类基础设施成熟了：

1. **训练 + 推理框架打通**：verl、vLLM、DeepSpeed 让 rollout、打分、梯度更新能流水线执行
2. **跨 tokenizer 蒸馏**：过去要求 Teacher 和 Student 用同一个 tokenizer，现在对齐技术成熟了
3. **MoE 提供更强的 Teacher**：MoE 让 Teacher 更大但推理成本可控，也支撑了大规模在线采样和打分

注意：MoE 是**工程支撑**，不是 OPD 数学优势的来源。它让 Teacher 更强、更便宜，但不是「为什么 OPD 比离线蒸馏好」的理由。

## 11. 论文速览（截至 2026-05）

| 论文 | 核心贡献 |
|:---|:---|
| **GKD** (Agarwal 2024) | OPD 源头：学生在自己生成的序列上接受 teacher 反馈 |
| **OPSD** (Zhao 2026) | 同一个模型同时当 teacher/student，teacher 多看 privileged info |
| **Pitfalls** (Zhu 2026) | OPD/OPSD 的风险：teacher 分布不可靠、loss 梯度不稳定、PI 不可内化 |
| **OGLS-SD** (Yang 2026) | 用 outcome reward 校准 token 级 teacher logits，修正反思偏置 |

共同结论：OPD/OPSD 不是银弹。真正优势是 **on-policy prefix + dense token-level feedback**；
真正风险是 teacher 分布不可靠、privileged information 无法在测试时被重建。

## 12. 论文全景：21 篇核心 OPD 论文分类（截至 2026-05）

awesome-on-policy-distillation 仓库整理了 OPD 领域的核心论文，按方法角色分为六大类：

#### 12.1 基础奠基（Foundations）

| 论文 | 核心贡献 | 类比 |
|:---|:---|:---|
| **MiniLLM** (Min 2023) | Reverse-KL 框架用于生成式 LM；**命名了 "On-Policy Distillation" 这个领域** | 定了基调：小模型不需要面面俱到 |
| **GKD** (Agarwal 2023) | 统一 on-/off-policy 混合训练 + 灵活散度选择 | 提供了通用的训练框架 |

#### 12.2 弥合差距（Gap-Bridging）— 当学生太弱怎么办

| 论文 | 核心贡献 |
|:---|:---|
| **Speculative KD** (2024) | 交替 teacher/student 采样，缓解学生 rollout 质量差的问题 |
| **Black-Box OPD / GAD** (2025) | Teacher 只有 API 没有 logits？用判别器当 reward 模型 |
| **SOD** (2026) | 按步骤级别差异重加权 teacher 指导，防止工具调用级联漂移 |
| **MAD-OPD** (2026) | 多智能体辩论共识当 teacher；扩展到 agent 任务 |
| **ROPD** (2026) | 黑盒 OPD，用 prompt 级 rubric（评分标准）蒸馏自 teacher-student 对比 |

#### 12.3 稳定性与目标设计（Stability）— OPD 不是银弹

| 论文 | 核心贡献 |
|:---|:---|
| **Veto** (2026) | 在 logit 空间引入中间目标分布，稳定训练 |
| **Entropy-Aware OPD** (2026) | 对高熵 teacher token 做 Forward-KL，保留输出多样性 |
| **ExOPD** (2026) | 把 OPD 看成稠密 KL 约束的 RL；reward scaling 让学生超越 teacher |
| **REOPOLD** (2026) | 放松模仿约束 + reward clipping + 基于熵的动态采样 |
| **PACED** (2026) | Pass-rate 加权，让学习聚焦在学生的「能力边界」上 |
| **Revisiting OPD** (2026) | 截断 reverse-KL + teacher top-K 支撑匹配；修复信号不均衡和 tokenizer 不匹配 |
| **Rock Tokens** (2026) | 高 loss token（可达 18%）在收敛后仍持续存在；mask 掉可以加速对齐 |
| **BRTS** (2026) | Teacher rollout 选最好的 N 个，避免低质量 teacher 信号 |
| **Prefix Teach, Suffix Fade** (2026) | 动态截断 dense supervision——当 teacher 的 margin 坍塌时停止教导后缀 |

#### 12.4 自蒸馏（Self-Distillation）— 不需要外部 Teacher

| 论文 | 核心贡献 |
|:---|:---|
| **OPSD** (Zhao 2026) | 同一模型 + privileged info = teacher；我们的 notebook 已覆盖 |
| **SDFT** (2026) | 演示条件自教学，用于持续学习 |
| **GATES** (2026) | 共识门控的非对称上下文自蒸馏，不需要标签或 reward |
| **RLSD** (2026) | GRPO 内的自蒸馏作为 token 级 credit assignment |
| **SDZero** (2026) | 生成器-修订器双角色：修订器把二元反馈转为 token 级监督 |
| **OPSDL** (2026) | 短上下文版本当 teacher，长上下文当 student |
| **OGLS-SD** (Yang 2026) | 我们的 notebook 已覆盖：用 outcome reward 校准 teacher logits |
| **OPHSD** (2026) | 把 privileged context 泛化为 harness 工作流（草稿-验证、计划-求解） |

#### 12.5 上下文与经验内化（Context Internalization）

| 论文 | 核心贡献 |
|:---|:---|
| **OPCD** (2026) | 上下文条件化的 teacher on student rollouts；蒸馏系统提示和经验知识 |
| **OEL** (2026) | 部署循环：用 OPCD 把交互轨迹固化为权重 |

#### 12.6 效率变体（Efficiency）

| 论文 | 核心贡献 |
|:---|:---|
| **Prefix OPD** (2026) | 只蒸馏推理前缀，FLOPs 降 2×-47× |
| **Lightning OPD** (2026) | 预计算 teacher log-probs；teacher-consistency 条件下 4× 加速 |
| **Nitrobrew** (2026) | Hidden-state teacher→student 传输 + 在线散度核；1.5-3× 吞吐 |
| **EffOPD** (2026) | 自适应外推当前更新步，~3× 训练加速 |

## 13. 分类维度：从两个角度理解 OPD 生态

awesome list 提供了两个有用的分类维度：

#### 13.1 按 Teacher 类型

| Teacher 类型 | 代表论文 | 你需要什么 |
|:---|:---|:---|
| **外部白盒** | MiniLLM, GKD, Veto, ExOPD, PACED, Revisiting OPD, Rock Tokens, ... | 能访问 Teacher 的 logits（全词表概率分布） |
| **外部黑盒** | GAD, OVD, ROPD | Teacher 只有 API（输入文本，输出文本），拿不到 logits |
| **自教师（privileged context）** | OPSD, SDFT, GATES, RLSD, SDZero, OGLS-SD, OPHSD | 不需要外部 Teacher，同一模型不同上下文 |
| **上下文条件化** | OPCD, OEL | Teacher 看到系统提示 / 交互历史，Student 不看到 |
| **多 Teacher / 生命周期** | MiMo MOPD, GLM-5, Qwen3, DeepSeek-V4 | 多个专家模型轮流或同时当 Teacher |

**关键决策：你能拿到 Teacher logits 吗？**
- 能 → 白盒路线（GKD, Veto, Entropy-Aware OPD）
- 不能 → 黑盒路线（GAD, OVD）或自蒸馏（OPSD, SDFT）

#### 13.2 按 主要目标

| 目标 | 代表论文 |
|:---|:---|
| **压缩 / 强到弱迁移** | MiniLLM, GKD, Qwen3, Prefix OPD, Rethinking OPD, Lightning OPD |
| **Post-RL 整合 / 技能合并** | MiMo MOPD, GLM-5, ExOPD, CoPD |
| **持续学习** | SDFT, OPCD, OEL |
| **RL 替代 / 增强** | SDPO, REOPOLD, RLSD, SDZero, OGLS-SD, PBSD, RLRT |
| **推理压缩** | OPSDC |
| **黑盒蒸馏** | GAD, OVD, ROPD |

注意：同一个方法可能同时属于多个类别。比如 OPSD 既是自蒸馏，也可以用于 RL 替代和推理压缩。

## 14. OPD 的工业落地

OPD 已经不是学术实验，而是主流 post-training 流水线的标准组件：

| 年份 | 系统 | OPD 用法 |
|:---|:---|:---|
| 2024 | **Gemma 2** | KD 作为 next-token prediction 的替代，训练 2B 和 9B 学生 |
| 2025 | **Qwen3** | 先 off-policy 再 on-policy 蒸馏，强到弱迁移 |
| 2025 | **GLM-4.5/4.6** | 多阶段 post-training，含专家模型迭代和 RL |
| 2026 | **MiMo-V2-Flash** | Multi-Teacher OPD (MOPD) 作为 post-training 阶段 |
| 2026 | **GLM-5** | On-policy 跨阶段蒸馏，恢复早期技能 |
| 2026 | **DeepSeek-V4** | 领域专家 SFT+GRPO → 统一模型通过 OPD 整合 |
| 2026 | **Baichuan-M3** | 任务 RL → 离线策略蒸馏 → 多 Teacher OPD |
| 2026 | **Nemotron-Cascade 2** (NVIDIA) | 级联 RL + 多领域 on-policy 蒸馏 |
| 2026 | **Qwen3.5-Omni** | 专家蒸馏 → privileged-input 自蒸馏对齐音频到文本 |
| 2026 | **HY-Embodied-0.5** | 32B → 2B on-policy 蒸馏；student rollout + teacher token 级监督 |
| 2026 | **Cursor Composer 2.5** | Hint-conditioned self-teacher OPD KL 叠加 RL |

**共同模式**：SFT → RL/GRPO → OPD 整合/压缩。OPD 通常在训练流水线的最后阶段，
用于把多个专家模型的能力合并到统一小模型中，或者作为 RL 之后的紧凑化步骤。

## 15. 从训练到上线：模型格式与部署工具

前面的 OPD 解决了「怎么训练出好模型」的问题。训练完成后，下一个问题就是：**怎么把它部署出去？**

这一节覆盖从「训练好的模型」到「能用的服务」之间会遇到的三个实践问题：模型存储格式、去哪里找模型、用什么工具部署。

### 模型格式

训练好的模型权重需要保存到文件。不同格式有不同的设计目标：

| 格式 | 特点 | 常见场景 |
|:-----|:-----|:---------|
| **Safetensors** | 安全（不含可执行代码）、加载快 | HuggingFace 标准格式，GPU 训练/推理 |
| **GGUF** | 支持多种量化等级、单文件、CPU 友好 | llama.cpp / ollama 本地推理 |
| **ONNX** | 跨框架、跨平台 | 部署到手机/浏览器/嵌入式设备 |
| **PyTorch (.bin/.pt)** | PyTorch 原生格式 | 研究、调试 |

为什么 HuggingFace 推荐用 Safetensors 而不是 PyTorch 原生的 .bin？

- PyTorch 的 `.bin` 底层用 `pickle` 序列化，加载时会执行任意代码——恶意模型文件可以在你的机器上运行任意命令
- Safetensors 只存数值，不含可执行代码，所以更安全
- Safetensors 的加载速度也更快（零拷贝 mmap）

In [ ]:
# 模型文件大小对比：同一模型，不同格式
# 以 Qwen2.5-7B 为例

params = 7e9  # 7B 参数

formats = [
    ("Safetensors (FP16)", params * 2 / 1e9, "GPU 训练/推理"),
    ("GGUF Q8_0", params * 1 / 1e9, "本地 CPU，高精度"),
    ("GGUF Q4_K_M", params * 0.56 / 1e9, "本地 CPU，平衡"),
    ("GGUF Q2_K", params * 0.31 / 1e9, "本地 CPU，省空间"),
    ("ONNX (FP16)", params * 2 / 1e9, "跨平台部署"),
]

print(f"Qwen2.5-7B 各格式大小:\n")
print(f"{'格式':<25} {'大小':>8} {'用途':<20}")
print("-" * 55)
for name, size_gb, usage in formats:
    print(f"{name:<25} {size_gb:>6.1f} GB  {usage}")

print(f"\n关键观察：GGUF Q4_K_M 把 7B 模型压到 ~4 GB，一张 8 GB 显存的卡就能跑")

### HuggingFace：模型之家

[HuggingFace](https://huggingface.co) 是目前最大的开源模型托管平台，几乎所有开源 LLM 都在这里发布。

实际使用时会遇到的核心操作：

**1. 找模型**：搜索 → 看 Model Card（模型说明文档）→ 看 Benchmarks（评测分数）→ 看社区评价

**2. 理解命名规则**：
```
Qwen/Qwen2.5-7B-Instruct-GGUF
 │      │       │    │       └─ 格式（GGUF）
 │      │       │    └─ 变体（Instruct = 对话微调版）
 │      │       └─ 参数量（7B = 70 亿参数）
 │      └─ 模型系列（Qwen2.5）
 └─ 组织/作者（Qwen 团队）
```

**3. 下载**：直接用 `huggingface-cli download` 或代码中 `from_pretrained()` 自动下载

| 后缀 | 含义 |
|:-----|:-----|
| `Base` | 预训练基座，没有经过对话微调 |
| `Instruct` | 经过 SFT + RLHF，可以直接对话 |
| `Chat` | 同 Instruct，不同团队的叫法 |
| `GGUF` | 量化格式，用于 llama.cpp / ollama |
| `AWQ` / `GPTQ` | GPU 量化格式，用于 vLLM |

In [ ]:
# 模拟：根据需求选择合适的模型文件

scenarios = [
    {
        "场景": "本地 Mac 上跑 7B 模型聊天",
        "推荐": "GGUF Q4_K_M + ollama",
        "理由": "ollama 一行命令安装，GGUF 对 Apple Silicon 友好，Q4_K_M 平衡精度和速度",
    },
    {
        "场景": "服务器上部署高并发 API",
        "推荐": "AWQ + vLLM",
        "理由": "vLLM 用 PagedAttention 做高吞吐，AWQ 在 GPU 上速度快精度高",
    },
    {
        "场景": "微调自己的模型",
        "推荐": "Safetensors (FP16) + QLoRA",
        "理由": "全精度基座 + LoRA 低秩微调，训练完合并回 Safetensors",
    },
    {
        "场景": "部署到手机端",
        "推荐": "ONNX + 量化",
        "理由": "ONNX 跨平台，手机端只能跑小模型（1-3B），需要极致压缩",
    },
]

print("=== 场景 → 推荐方案 ===\n")
for s in scenarios:
    print(f"场景: {s['场景']}")
    print(f"  推荐: {s['推荐']}")
    print(f"  理由: {s['理由']}")
    print()

### 部署工具选型

| 工具 | 定位 | 量化支持 | 适用场景 |
|:-----|:-----|:---------|:---------|
| **llama.cpp** | C++ 推理引擎 | GGUF | CPU / Apple Silicon / 单卡 GPU |
| **ollama** | llama.cpp 的简化包装 | GGUF | 本地开发、快速体验 |
| **vLLM** | GPU 高吞吐推理 | AWQ / GPTQ / FP16 | 生产环境、高并发 API |
| **SGLang** | 结构化生成优化 | 多种 | 需要严格 JSON 输出的场景 |
| **TensorRT-LLM** | NVIDIA 优化推理 | FP8 / INT8 | 追求极致性能的 GPU 部署 |

选择逻辑：
```
有没有 GPU？
  ├─ 没有 → llama.cpp / ollama（GGUF）
  └─ 有
      ├─ 追求吞吐量 → vLLM
      ├─ 追求结构化输出 → SGLang
      └─ 追求极致性能 → TensorRT-LLM
```

## 小结

### 核心原理（Sections 1-9）

1. ✅ **知识蒸馏** = 大模型教小模型
2. ✅ **离线蒸馏** = 背老师写的范文 → prefix 来自老师/数据集
3. ✅ **Exposure Bias** = 训练 prefix 和推理 prefix 不一致 → 推理时走偏就崩溃
4. ✅ **OPD** = Student 自己 rollout，Teacher 在 Student 的 prefix 上给反馈
5. ✅ **Forward KL**（离线蒸馏）= Mode-Covering → 小模型什么都学，什么都不精
6. ✅ **Reverse KL**（OPD）= Mode-Seeking → 小模型专攻一门
7. ✅ **OPSD** = 不需要外部 Teacher，同一个模型的开卷版教闭卷版
8. ✅ **三种粒度**：sampled-token < top-k < full-vocab
9. ✅ **三种 KL 估计器**：k1（无偏可为负）、k2（有偏永远正）、k3（无偏永远正，推荐）
11. ✅ **模型格式**：Safetensors（安全加载）、GGUF（量化推理）、ONNX（跨平台）
12. ✅ **HuggingFace**：开源模型的中心，理解命名规则是找模型的第一步
13. ✅ **部署工具**：无 GPU 用 ollama，有 GPU 用 vLLM，追求极致用 TensorRT-LLM

### 工程与生态（Sections 10-15）

10. ✅ **工程基础设施**：verl + vLLM + DeepSpeed + 跨 tokenizer 对齐让 OPD 落地
11. ✅ **论文速览**：GKD / OPSD / Pitfalls / OGLS-SD 四篇核心论文
12. ✅ **论文全景**：21 篇核心论文分六大类（基础 / 弥合差距 / 稳定性 / 自蒸馏 / 上下文内化 / 效率）
13. ✅ **分类维度**：按 Teacher 类型（白盒/黑盒/自教师/上下文/多教师）和按目标（压缩/整合/持续学习/RL替代）
14. ✅ **工业落地**：Qwen3 / DeepSeek-V4 / GLM-5 / MiMo / Nemotron 等均在用，标准流水线 = SFT → RL → OPD
15. ✅ **工具框架**：TRL（入门）/ veRL（生产）/ NeMo-RL（跨 tokenizer）/ nano-opd（学习）

### 一句话总结

离线蒸馏 = 学生背老师写的标准答案；OPD = 学生自己写，老师逐句批改。
背答案的学生一到实战就暴露不足，被批改过的学生才真正学会了。

OPD 不是银弹——真正优势是 **on-policy prefix + dense token-level feedback**，真正风险是 **teacher 分布不可靠 + PI 无法内化**。
选哪条路取决于你的约束：有 logits → 白盒，只有 API → 黑盒，没有外部 teacher → 自蒸馏。

### 参考资料

- awesome-on-policy-distillation: https://github.com/chrisliu298/awesome-on-policy-distillation
- OPD Survey: https://arxiv.org/abs/2604.00626
- MiniLLM: https://arxiv.org/abs/2306.08543
- GKD: https://arxiv.org/abs/2306.13649
- ExOPD: https://arxiv.org/abs/2602.12125
- OPSD: https://arxiv.org/abs/2601.18734
- Pitfalls (Zhu et al.): https://arxiv.org/abs/2605.11182
- OGLS-SD: https://arxiv.org/abs/2605.12400
- Thinking Machines blog: https://thinkingmachines.ai/blog/on-policy-distillation/
- TRL DistillationTrainer: https://huggingface.co/docs/trl